In [ ]:
!pip install lightning torchinfo optuna requests pyyaml dukascopy-python

In [ ]:
def is_running_in_colab():
    try:
        import google.colab
        return True
    except ImportError:
        return False

if is_running_in_colab():
    %cd /content/drive/MyDrive/02_tcn_fx_predictor/src

import sys
import os
parent_dir = os.path.abspath('..')
if parent_dir not in sys.path:
    sys.path.append(parent_dir)

import datetime as dt
import requests
#import dukascopy_python
#from dukascopy_python.instruments import INSTRUMENT_FX_MAJORS_EUR_USD

import polars as pl
from polars import col as c
import pandas as pd
import numpy as np
from src.configs.configs import BASE_DIR, DATA_DIR, DATA_DIR_RAW, DATA_DIR_PROCESSED, DATA_DIR_FEATURES_TARGETS
from collections import defaultdict

import torch
import torch.nn as nn
import lightning as L
from torchinfo import summary
from torch.utils.data import TensorDataset, DataLoader
from src.models.TCN import TemporalConvNet, LightningTCN
from src.data.processor import data_processing, sliding_window, feature_engineering, train_test_split, sliding_window
from src.training.trainer import trainer, objective
from src.evaluation.evaluator import evaluation

from sklearn.preprocessing import StandardScaler
from lightning.pytorch.callbacks import EarlyStopping, ModelCheckpoint
import optuna
import yaml
from sklearn.metrics import precision_score, recall_score, classification_report

#from optuna.integration import PyTorchLightningPruningCallback

In [ ]:
#lf=download()
#lf = data_processing()
lf = feature_engineering()
X_train,y_train, X_val, y_val, X_test, y_test = train_test_split(lf)

scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_val = scaler.transform(X_val)
X_test = scaler.transform(X_test)

sql=48
X_train = sliding_window(X_train, sql).transpose(0,2,1)
y_train = y_train[sql-1:,:]
X_val = sliding_window(X_val, sql).transpose(0,2,1)
y_val = y_val[sql-1:,:]
X_test = sliding_window(X_test, sql).transpose(0,2,1)
y_test = y_test[sql-1:,:]
y_train = y_train[:,2]
y_val = y_val[:,2]
y_test = y_test[:,2]

In [ ]:
with open(BASE_DIR/'src/configs/params_configs.yaml', 'r', encoding='utf-8') as file:
    config = yaml.safe_load(file)

epochs=50
num_channels = config['model']['num_channels']
kernel_size = config['model']['kernel_size']
dropout = config['model']['dropout']

epochs = config['train']['epochs']
batch_size = config['train']['batch_size']
loss_name = config['train']['loss']['name']
use_class_weight = config['train']['loss']['use_class_weight']

lr = config['train']['optimizer']['lr']

num_inputs = X_train.shape[1]

if use_class_weight:
    class_counts = np.bincount(y_train, minlength=3)
    weights = len(y_train) / (3 * class_counts)
    class_weights = torch.tensor(weights, dtype=torch.float32)
    criterion = nn.CrossEntropyLoss(weight=class_weights)
else:
    criterion = nn.CrossEntropyLoss()

opt_name = config['train']['optimizer']['name'].lower()
optimizer_ = {'adam': torch.optim.Adam, 'sgd': torch.optim.SGD}[opt_name]

tcn_model = TemporalConvNet(num_inputs=num_inputs, num_channels=num_channels, kernel_size=kernel_size, dropout=dropout, num_outputs=3)

In [ ]:
data = pl.scan_parquet(DATA_DIR_FEATURES_TARGETS/'features_targets').sort('timestamp')
target_cols = [f'd{n}' for n in [10,30,60,180]]
X = data.select(pl.all().exclude(target_cols+['timestamp'])).collect().to_numpy()
y = data.select(target_cols).collect().to_numpy()[:,2]

cutoff2 = int( y.shape[0] *0.8 )
X = X[:cutoff2]
y = y[:cutoff2]

study = optuna.create_study(direction='minimize')
study.optimize(lambda trial: objective(trial, X, y), n_trials=20)

In [ ]:
trainer(tcn_model, X_train, y_train, X_val, y_val, criterion, optimizer_, lr, epochs)

In [ ]:
bst_model_path = BASE_DIR/'checkpoints/best_model-v6.ckpt'
evaluation(X_test, y_test, bst_model_path ,num_channels, epochs,num_inputs,criterion,lr,optimizer_ )

In [ ]:
X_test = torch.tensor(X_test, dtype=torch.float32).to('cuda')
y_test = torch.tensor(y_test)


model = TemporalConvNet(num_inputs=num_inputs, num_channels=num_channels, kernel_size=kernel_size, dropout=dropout, num_outputs=3)
lightning_model = LightningTCN(model=model, lr=lr, criterion=criterion, optimizer_=optimizer_)
bst_model_path = BASE_DIR/'checkpoints/best_model-v6.ckpt'
checkpoint = torch.load(bst_model_path, map_location='cuda')
lightning_model.load_state_dict(checkpoint['state_dict'])
lightning_model.to('cuda')
lightning_model.eval()

with torch.no_grad():
    logits = lightning_model(X_test)

y_preds = logits.cpu().numpy()
y_preds = np.argmax(y_preds, axis=1)

p_macro = precision_score(y_test, y_preds, average='macro')
print(p_macro)
r_macro = recall_score(y_test, y_preds, average='macro')
print(r_macro)
print(classification_report(y_test, y_preds))